# Add INT16 activation Q-DQ to an OGA 2-bit model

Adds static **UINT16 activation** / **UINT8 weight** Q-DQ on top of an OGA-translated
`MatMulNBits` (2-bit) model, producing a **W2A16** model for AMD Ryzen AI NPU-transformer
deployment. The packed 2-bit weights are left untouched.

See [add_int16_qdq.py](add_int16_qdq.py) for the standalone CLI; this notebook mirrors it.


## 1. Inputs


In [ ]:
OGA_OUT = "/root/oga_out/phi4_qad_uint2"  # output of translate_2bit_oga.ipynb
W2A16_OUT = "/root/oga_out/phi4_qad_uint2_int16"  # INT16-QDQ output
BACKEND = "quark"  # 'quark' (recommended) or 'ort'

## 2. Run the tool

`quark` is the default backend (Ryzen AI ready). Add `--npu-transformer` for power-of-two
INT16 scales. Use `--backend ort` for the dependency-light ORT fallback.


In [ ]:
!python add_int16_qdq.py -i "$OGA_OUT" -o "$W2A16_OUT" --backend "$BACKEND" \
    --calib-samples 32 --seq-len 512

Expected summary (phi-4 QAD v6, full model):
```
QuantizeLinear=442 DequantizeLinear=525 MatMulNBits=200 (preserved)
zero-point dtypes: ['UINT16', 'UINT8']
```
`MatMulNBits` count is unchanged (weights stay 2-bit); activations feeding them are now UINT16 Q-DQ.


## 3. Inspect the graph (before/after)


In [ ]:
import collections

import onnx


def op_hist(path):
    m = onnx.load(path, load_external_data=False)
    return collections.Counter(n.op_type for n in m.graph.node)


before = op_hist(f"{OGA_OUT}/model.onnx")
after = op_hist(f"{W2A16_OUT}/model.onnx")
print("MatMulNBits  before/after:", before.get("MatMulNBits", 0), "/", after.get("MatMulNBits", 0))
print("QuantizeLinear   after   :", after.get("QuantizeLinear", 0))
print("DequantizeLinear after   :", after.get("DequantizeLinear", 0))

## 4. (optional) Perplexity comparison

Compare WikiText-2 perplexity of the W2A16 model vs the pre-Q-DQ (W2A-float) baseline to
quantify the INT16 activation cost. Use the `wikitext_ppl_oga.py` helper (raw ORT, CPU):

```bash
python wikitext_ppl_oga.py <model_dir> 2048 0
```

For phi-4 QAD v6: pre-Q-DQ baseline PPL = **12.70**; the W2A16 delta is reported in the README.
